# 信噪 · 4 路对比评估 (Kaggle)

跑完训练后用这个 notebook 做对比，得到简历能写的关键数字。

对比策略：
1. **deepseek-base** —— DeepSeek-V3 纯 LLM（无 RAG、无工具）
2. **deepseek-agent** —— 当前完整 agent（RAG + 5 工具 + injection guard）
3. **qwen-base** —— Qwen2.5-1.5B-Instruct 原生（无微调）
4. **qwen-lora** —— Qwen2.5-1.5B + 你刚训出来的 LoRA adapter

全部用同一个 LLM-as-judge（4 维度评分）给所有回复打分，结果保存到 `evals/compare_report.md`。

## 准备
1. **加入项目代码 Dataset**（如未上传请打包成 Dataset 加进来）
2. **加入训练后的 adapter** —— 把训练 notebook 的 Output `qwen-1.5b-xinzao-lora/` 也作为 Dataset 加到本 notebook 输入
3. **配置 Secret**：`DEEPSEEK_API_KEY`、`TAVILY_API_KEY`（左侧 Add-ons → Secrets）
4. **Settings**：Accelerator = GPU T4/P100；Internet = On

In [ ]:
# 1. 安装运行时 + 训练推理依赖
!pip install -q -U openai python-dotenv tavily-python sentence-transformers rank-bm25
!pip install -q -U transformers peft accelerate

In [ ]:
# 2. 设置环境变量 + 找路径
import os, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["DEEPSEEK_API_KEY"] = secrets.get_secret("DEEPSEEK_API_KEY")
os.environ["TAVILY_API_KEY"] = secrets.get_secret("TAVILY_API_KEY")

# 自动找代码目录（包含 evals/ 和 src/）
INPUT_ROOT = Path("/kaggle/input")
code_root = None
for p in INPUT_ROOT.rglob("run_compare.py"):
    code_root = p.parents[1]
    break
assert code_root, "找不到项目代码 Dataset（应包含 evals/run_compare.py）"
print(f"代码目录: {code_root}")

# 把代码复制到 /kaggle/working 以便写入 trace/report
WORK_DIR = Path("/kaggle/working/repo")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(code_root, WORK_DIR)
os.chdir(WORK_DIR)
print(f"工作目录: {WORK_DIR}")

# 找 LoRA adapter
adapter_candidates = list(INPUT_ROOT.rglob("adapter_config.json"))
assert adapter_candidates, "找不到 LoRA adapter（应有 adapter_config.json）"
ADAPTER_PATH = adapter_candidates[0].parent
print(f"LoRA adapter: {ADAPTER_PATH}")

In [ ]:
# 3. 跑全四路对比
import subprocess, sys

cmd = [
    sys.executable, "evals/run_compare.py",
    "--strategies", "all",
    "--qwen-base-name", "Qwen/Qwen2.5-1.5B-Instruct",
    "--qwen-lora-path", str(ADAPTER_PATH),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 4. 看报告
from IPython.display import Markdown, display
report = Path("evals/compare_report.md").read_text(encoding="utf-8")
display(Markdown(report))

In [ ]:
# 5. 拷一份报告 + 结果到 Kaggle Output，方便下载
import shutil
shutil.copy("evals/compare_report.md", "/kaggle/working/compare_report.md")
shutil.copy("evals/compare_results.json", "/kaggle/working/compare_results.json")
print("输出文件已拷到 /kaggle/working/，notebook 结束后可从 Output 下载")